# MathQA Test

This notebook runs repeated sampling on `math_qa` with a parameterized vLLM model.

What it does:

- loads a parameterized number of MathQA examples
- prompts the model to end with `{"Final Answer": "a"}`
- parses everything before that JSON into a `"Reasoning"` field
- samples each example multiple times with vLLM
- saves raw sample-level outputs incrementally
- computes per-example and overall metrics such as answer entropy, fraction correct, and reasoning token counts

The notebook is intentionally step-by-step so it is easy to inspect intermediate state and debug.


## 1. Setup

Edit `CONFIG` below if you want to change the model, number of examples, number of samples per example, output location, or vLLM settings.


In [ ]:
from __future__ import annotations

import json
import math
import random
import re
import sys
from collections import Counter
from datetime import datetime
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from datasets import load_dataset
from IPython.display import JSON, display
from tqdm.auto import tqdm
from transformers import AutoTokenizer

try:
    from vllm import LLM, SamplingParams
except ImportError as exc:
    raise RuntimeError(
        "This notebook requires vLLM in the active kernel. "
        "Switch to a kernel/environment with `vllm` installed before running the generation cells."
    ) from exc

CONFIG: dict[str, Any] = {
    "model_name": "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B",
    "dataset_name": "math_qa",
    "dataset_config_name": None,
    "split": "test",
    "num_examples": 5,
    "samples_per_example": 20,
    "sample_batch_size": 100,
    "seed": 7,
    "temperature": 0.7,
    "top_p": 0.95,
    "max_tokens": 5000,
    "tensor_parallel_size": 1,
    "gpu_memory_utilization": 0.9,
    "trust_remote_code": True,
    "use_chat_template": True,
    "base_output_dir": "/playpen-ssd/smerrill/commitment/notebooks/runs",
    "run_name": None,
    "resume_from_existing": True,
}

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])

def slugify(value: str) -> str:
    value = value.replace("/", "__")
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value)
    return value.strip("_")

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
resolved_run_name = CONFIG["run_name"] or f"mathqa_{slugify(CONFIG['model_name'])}_{timestamp}"
BASE_OUTPUT_DIR = Path(CONFIG["base_output_dir"])
RUN_DIR = BASE_OUTPUT_DIR / resolved_run_name
RUN_DIR.mkdir(parents=True, exist_ok=True)

CONFIG_TO_SAVE = dict(CONFIG)
CONFIG_TO_SAVE["run_name"] = resolved_run_name
CONFIG_TO_SAVE["run_dir"] = str(RUN_DIR)

config_path = RUN_DIR / "run_config.json"
config_path.write_text(json.dumps(CONFIG_TO_SAVE, indent=2), encoding="utf-8")

print(f"Python: {sys.version.split()[0]}")
print(f"Run directory: {RUN_DIR}")
display(JSON(CONFIG_TO_SAVE))


## 2. Prompting and parsing helpers

The model is asked to produce free-form reasoning first and then end with a JSON object of the form `{"Final Answer": "a"}`.

The parser below turns each raw completion into:

- `{"Reasoning": ... , "Final Answer": ...}`
- plus a few debug fields such as parse success and the raw completion


In [ ]:
SYSTEM_PROMPT = '''
You are solving a multiple-choice math problem.

Think through the problem carefully. Your response must end with a JSON object on the final line
that uses exactly this schema:
{"Final Answer": "a"}

Rules:
- Put all reasoning before the JSON object.
- Do not use markdown code fences.
- Do not add any text after the JSON object.
- The final answer should be the single best option letter from a, b, c, d, or e.
'''

USER_PROMPT_TEMPLATE = '''
Solve this MathQA multiple-choice problem.

Problem and options:
{question}
'''

FINAL_ANSWER_JSON_RE = re.compile(
    r'\{\s*"Final Answer"\s*:\s*(?P<value>"(?:\\.|[^"\\])*"|[^{}]+?)\s*\}',
    flags=re.DOTALL,
)
OPTION_RE = re.compile(
    r"([A-Ea-e])\s*[\)\].:]\s*(.*?)(?=(?:\s*,\s*[A-Ea-e]\s*[\)\].:])|$)",
    flags=re.DOTALL,
)


def parse_mathqa_options(options_text: str) -> dict[str, str]:
    cleaned = re.sub(r"\s+", " ", str(options_text or "")).strip()
    if not cleaned:
        return {}
    matches = list(OPTION_RE.finditer(cleaned))
    option_map: dict[str, str] = {}
    for match in matches:
        label = match.group(1).lower()
        text = match.group(2).strip().strip(",").strip()
        if text:
            option_map[label] = text
    return option_map


def format_options_block(options_text: str) -> str:
    option_map = parse_mathqa_options(options_text)
    if not option_map:
        return re.sub(r"\s+", " ", str(options_text or "")).strip()
    return "\n".join(f"{label}) {text}" for label, text in option_map.items())


def build_combined_question(problem: str, options_text: str) -> str:
    problem = re.sub(r"\s+", " ", str(problem or "")).strip()
    options_block = format_options_block(options_text)
    return f"{problem}\n\nOptions:\n{options_block}".strip()


def render_prompt(question: str, tokenizer: AutoTokenizer, use_chat_template: bool = True) -> str:
    user_prompt = USER_PROMPT_TEMPLATE.format(question=question.strip())
    if use_chat_template and getattr(tokenizer, "chat_template", None):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT.strip()},
            {"role": "user", "content": user_prompt.strip()},
        ]
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            pass

    return (
        f"System:\n{SYSTEM_PROMPT.strip()}\n\n"
        f"User:\n{user_prompt.strip()}\n\n"
        "Assistant:\n"
    )


def clean_reasoning(text: str) -> str:
    text = (text or "").strip()
    text = re.sub(r"^<think>\s*", "", text)
    text = re.sub(r"\s*</think>\s*$", "", text)
    return text.strip()


def coerce_final_answer_value(raw_value: str) -> str | None:
    raw_value = (raw_value or "").strip().rstrip(",").strip()
    if not raw_value:
        return None

    try:
        parsed_value = json.loads(raw_value)
    except json.JSONDecodeError:
        if raw_value.startswith('"') and raw_value.endswith('"') and len(raw_value) >= 2:
            value = raw_value[1:-1]
        else:
            value = raw_value

        value = value.replace(r"\\n", " ")
        value = value.replace(r"\\t", " ")
        value = value.replace(r"\\r", " ")
        value = value.replace(r'\\"', '"')
        value = value.replace(r"\\\\", r"\\")
        value = re.sub(r"\s+", " ", value)
        return value.strip() or None

    return str(parsed_value).strip() or None


def parse_model_output(raw_text: str) -> dict[str, Any]:
    raw_text = raw_text or ""
    matches = list(FINAL_ANSWER_JSON_RE.finditer(raw_text))
    if matches:
        match = matches[-1]
        candidate = match.group(0)
        reasoning = clean_reasoning(raw_text[: match.start()])

        try:
            parsed_json = json.loads(candidate)
        except json.JSONDecodeError as exc:
            final_answer = coerce_final_answer_value(match.group("value"))
            return {
                "Reasoning": reasoning,
                "Final Answer": final_answer,
                "Parse Error": f"{type(exc).__name__}: {exc}",
                "parse_success": False,
                "raw_completion": raw_text,
                "matched_final_answer_block": candidate,
            }

        final_answer = str(parsed_json.get("Final Answer", "")).strip() or None
        return {
            "Reasoning": reasoning,
            "Final Answer": final_answer,
            "Parse Error": None,
            "parse_success": True,
            "raw_completion": raw_text,
            "matched_final_answer_block": candidate,
        }

    fallback = re.search(r"Final Answer\s*[:=]\s*(?P<answer>.+)$", raw_text, flags=re.DOTALL)
    if fallback:
        reasoning = clean_reasoning(raw_text[: fallback.start()])
        final_answer = fallback.group("answer").strip().strip("`").strip().strip('"').strip()
        return {
            "Reasoning": reasoning,
            "Final Answer": final_answer or None,
            "Parse Error": "No valid final JSON object found; used fallback Final Answer parsing.",
            "parse_success": False,
            "raw_completion": raw_text,
            "matched_final_answer_block": None,
        }

    return {
        "Reasoning": clean_reasoning(raw_text),
        "Final Answer": None,
        "Parse Error": "No final answer JSON object or fallback pattern found.",
        "parse_success": False,
        "raw_completion": raw_text,
        "matched_final_answer_block": None,
    }


def extract_gold_choice(value: Any) -> str:
    match = re.search(r"\b([A-Ea-e])\b", str(value or ""))
    if not match:
        raise ValueError(f"Could not find MathQA final answer in: {value!r}")
    return match.group(1).lower()


def extract_choice_label(text: Any, option_map: dict[str, str] | None = None) -> str | None:
    if text is None:
        return None
    value = str(text).strip()
    if not value:
        return None

    patterns = [
        r"(?i)\boption\s*([a-e])\b",
        r"(?i)\banswer(?:\s+is)?\s*[:=-]?\s*([a-e])\b",
        r"(?i)^\(?\s*([a-e])\s*\)?$",
        r"(?i)^\(?\s*([a-e])\s*[\)\].:-].*$",
    ]
    for pattern in patterns:
        match = re.search(pattern, value)
        if match:
            return match.group(1).lower()

    compact = re.sub(r"[^A-Za-z]", "", value).lower()
    if compact in {"a", "b", "c", "d", "e"}:
        return compact

    if option_map:
        normalized_value = normalize_answer(value)
        for label, option_text in option_map.items():
            if normalized_value == normalize_answer(option_text):
                return label

    return None


def normalize_answer(text: Any, option_map: dict[str, str] | None = None) -> str | None:
    label = extract_choice_label(text, option_map=option_map)
    if label is not None:
        return label
    if text is None:
        return None

    value = str(text).strip()
    if not value:
        return None

    value = value.strip("`").strip()
    value = value.strip('"').strip("'")
    value = value.replace("$", "")
    value = value.replace(",", "")
    value = value.rstrip(".")
    value = re.sub(r"\s+", " ", value)
    return value.lower() or None


def reasoning_token_count(reasoning: str, tokenizer: AutoTokenizer) -> int:
    if not reasoning:
        return 0
    token_ids = tokenizer.encode(reasoning, add_special_tokens=False)
    return len(token_ids)


def shannon_entropy_bits(values: list[str]) -> float:
    if not values:
        return float("nan")
    counts = Counter(values)
    total = sum(counts.values())
    probabilities = [count / total for count in counts.values()]
    return float(-sum(p * math.log2(p) for p in probabilities if p > 0))


def append_jsonl(path: Path, records: list[dict[str, Any]]) -> None:
    if not records:
        return
    with path.open("a", encoding="utf-8") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\\n")


## 3. Load and subsample MathQA examples

The notebook randomly samples examples from the requested split, keeps the original problem text plus options, and stores the gold answer as the option letter.


In [ ]:
load_dataset_kwargs = {
    "path": CONFIG["dataset_name"],
    "split": CONFIG["split"],
    "trust_remote_code": CONFIG["trust_remote_code"],
}
if CONFIG["dataset_config_name"] is not None:
    load_dataset_kwargs["name"] = CONFIG["dataset_config_name"]

dataset = load_dataset(**load_dataset_kwargs)

if CONFIG["num_examples"] > len(dataset):
    raise ValueError(
        f"Requested {CONFIG['num_examples']} examples, but split only has {len(dataset)} rows."
    )

rng = np.random.default_rng(CONFIG["seed"])
selected_indices = rng.choice(len(dataset), size=CONFIG["num_examples"], replace=False)
selected_indices = [int(index) for index in selected_indices]

selected_rows = dataset.select(selected_indices)
selected_examples = pd.DataFrame(
    {
        "example_id": list(range(len(selected_rows))),
        "dataset_index": selected_indices,
        "problem": selected_rows["Problem"],
        "options": selected_rows["options"],
        "category": selected_rows["category"],
        "gold_final_answer": [extract_gold_choice(value) for value in selected_rows["correct"]],
    }
)
selected_examples["question"] = selected_examples.apply(
    lambda row: build_combined_question(row["problem"], row["options"]),
    axis=1,
)
selected_examples["gold_option_text"] = selected_examples.apply(
    lambda row: parse_mathqa_options(row["options"]).get(row["gold_final_answer"]),
    axis=1,
)
selected_examples["gold_answer_full"] = selected_examples.apply(
    lambda row: f"{row['gold_final_answer']}) {row['gold_option_text']}",
    axis=1,
)
selected_examples["gold_final_answer_normalized"] = selected_examples["gold_final_answer"]

selected_examples_path = RUN_DIR / "selected_examples.csv"
selected_examples.to_csv(selected_examples_path, index=False)

print(f"Loaded split size: {len(dataset)}")
print(f"Selected example indices: {selected_indices}")
display(selected_examples[["example_id", "dataset_index", "category", "gold_answer_full", "question"]])


## 4. Initialize the tokenizer and vLLM model, then preview one sample

This preview cell is useful before launching the full 100 x 100 run.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_name"],
    trust_remote_code=CONFIG["trust_remote_code"],
)

llm = LLM(
    model=CONFIG["model_name"],
    tensor_parallel_size=CONFIG["tensor_parallel_size"],
    gpu_memory_utilization=CONFIG["gpu_memory_utilization"],
    trust_remote_code=CONFIG["trust_remote_code"],
)

preview_sampling_params = SamplingParams(
    temperature=CONFIG["temperature"],
    top_p=CONFIG["top_p"],
    max_tokens=CONFIG["max_tokens"],
)

preview_row = selected_examples.iloc[0]
preview_prompt = render_prompt(
    question=preview_row["question"],
    tokenizer=tokenizer,
    use_chat_template=CONFIG["use_chat_template"],
)

preview_output = llm.generate([preview_prompt], preview_sampling_params)[0].outputs[0].text
parsed_preview = parse_model_output(preview_output)
preview_structured = {
    "Reasoning": parsed_preview["Reasoning"],
    "Final Answer": parsed_preview["Final Answer"],
    "Parse Error": parsed_preview["Parse Error"],
}

print("Preview question:")
print(preview_row["question"])
print("\nPreview raw completion:")
print(preview_output)
print("\nPreview parsed output:")
print(json.dumps(preview_structured, indent=2, ensure_ascii=False))
print("\nGold final answer:", preview_row["gold_final_answer"])


## 5. Full repeated sampling run with incremental saves

This cell:

- samples each selected example `samples_per_example` times
- saves records to JSONL as they are produced
- also writes a flattened CSV snapshot at the end

If `resume_from_existing` is `True`, rerunning this cell will continue from an existing JSONL file in the same run directory.


In [ ]:
SAMPLE_JSONL_PATH = RUN_DIR / "sample_records.jsonl"
SAMPLE_CSV_PATH = RUN_DIR / "sample_records.csv"


def load_existing_records(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


def run_sampling_experiment(
    examples_df: pd.DataFrame,
    tokenizer: AutoTokenizer,
    llm: LLM,
    config: dict[str, Any],
    run_dir: Path,
) -> pd.DataFrame:
    existing_df = load_existing_records(SAMPLE_JSONL_PATH)
    completed_counts: dict[int, int] = {}

    if not existing_df.empty and config["resume_from_existing"]:
        completed_counts = (
            existing_df.groupby("example_id")["sample_id"].count().astype(int).to_dict()
        )
        print(f"Resuming from existing file: {SAMPLE_JSONL_PATH}")
        print(f"Existing records: {len(existing_df)}")
    elif SAMPLE_JSONL_PATH.exists() and not config["resume_from_existing"]:
        raise FileExistsError(
            f"{SAMPLE_JSONL_PATH} already exists. "
            "Set CONFIG['resume_from_existing'] = True or pick a new run_name."
        )

    sampling_params = SamplingParams(
        temperature=config["temperature"],
        top_p=config["top_p"],
        max_tokens=config["max_tokens"],
    )

    all_new_records: list[dict[str, Any]] = []

    for _, row in tqdm(examples_df.iterrows(), total=len(examples_df), desc="Examples"):
        example_id = int(row["example_id"])
        already_done = int(completed_counts.get(example_id, 0))
        remaining = config["samples_per_example"] - already_done

        if remaining <= 0:
            continue

        prompt = render_prompt(
            question=row["question"],
            tokenizer=tokenizer,
            use_chat_template=config["use_chat_template"],
        )
        gold_final_normalized = row["gold_final_answer_normalized"]
        option_map = parse_mathqa_options(row["options"])

        next_sample_id = already_done
        while remaining > 0:
            current_batch_size = min(config["sample_batch_size"], remaining)
            prompts = [prompt] * current_batch_size
            request_outputs = llm.generate(prompts, sampling_params)

            batch_records: list[dict[str, Any]] = []
            for request_output in request_outputs:
                raw_completion = request_output.outputs[0].text
                parsed = parse_model_output(raw_completion)
                final_answer_normalized = normalize_answer(parsed["Final Answer"], option_map=option_map)
                reasoning = parsed["Reasoning"]
                reasoning_tokens = reasoning_token_count(reasoning, tokenizer)

                record = {
                    "run_name": resolved_run_name,
                    "model_name": config["model_name"],
                    "example_id": example_id,
                    "dataset_index": int(row["dataset_index"]),
                    "sample_id": int(next_sample_id),
                    "problem": row["problem"],
                    "options": row["options"],
                    "category": row["category"],
                    "question": row["question"],
                    "gold_answer_full": row["gold_answer_full"],
                    "gold_final_answer": row["gold_final_answer"],
                    "gold_final_answer_normalized": gold_final_normalized,
                    "raw_completion": raw_completion,
                    "reasoning": reasoning,
                    "final_answer": parsed["Final Answer"],
                    "final_answer_normalized": final_answer_normalized,
                    "parse_success": bool(parsed["parse_success"]),
                    "parse_error": parsed["Parse Error"],
                    "matched_final_answer_block": parsed["matched_final_answer_block"],
                    "is_correct": bool(
                        final_answer_normalized is not None
                        and final_answer_normalized == gold_final_normalized
                    ),
                    "reasoning_tokens": int(reasoning_tokens),
                    "completion_chars": int(len(raw_completion)),
                }
                batch_records.append(record)
                next_sample_id += 1

            append_jsonl(SAMPLE_JSONL_PATH, batch_records)
            all_new_records.extend(batch_records)
            remaining -= current_batch_size

    final_df = load_existing_records(SAMPLE_JSONL_PATH)
    final_df.to_csv(SAMPLE_CSV_PATH, index=False)

    print(f"Sample JSONL: {SAMPLE_JSONL_PATH}")
    print(f"Sample CSV:   {SAMPLE_CSV_PATH}")
    print(f"New records generated in this run: {len(all_new_records)}")
    print(f"Total saved records: {len(final_df)}")
    return final_df


sample_df = run_sampling_experiment(
    examples_df=selected_examples,
    tokenizer=tokenizer,
    llm=llm,
    config=CONFIG,
    run_dir=RUN_DIR,
)

display(sample_df.head(3))


## 6. Compute per-example and overall metrics

Core metrics included here:

- `final_answer_entropy_bits`: entropy of the normalized final answer distribution for one example
- `fraction_correct`: fraction of samples that exactly match the normalized MathQA gold answer
- `mean_reasoning_tokens`: average number of reasoning tokens before the final answer JSON


In [ ]:
if "sample_df" not in globals() or sample_df.empty:
    sample_df = load_existing_records(SAMPLE_JSONL_PATH)


def summarize_example(group: pd.DataFrame) -> pd.Series:
    valid_answers = [
        answer for answer in group["final_answer_normalized"].dropna().tolist() if str(answer).strip()
    ]
    return pd.Series(
        {
            "dataset_index": int(group["dataset_index"].iloc[0]),
            "question": group["question"].iloc[0],
            "gold_final_answer": group["gold_final_answer"].iloc[0],
            "num_samples": int(len(group)),
            "parse_success_rate": float(group["parse_success"].mean()),
            "fraction_correct": float(group["is_correct"].mean()),
            "mean_reasoning_tokens": float(group["reasoning_tokens"].mean()),
            "median_reasoning_tokens": float(group["reasoning_tokens"].median()),
            "std_reasoning_tokens": float(group["reasoning_tokens"].std(ddof=0)),
            "final_answer_entropy_bits": float(shannon_entropy_bits(valid_answers)),
            "unique_final_answers": int(len(set(valid_answers))),
        }
    )


example_metrics = (
    sample_df.groupby("example_id", sort=True, dropna=False)
    .apply(summarize_example)
    .reset_index()
)

example_metrics_path = RUN_DIR / "example_metrics.csv"
example_metrics.to_csv(example_metrics_path, index=False)

overall_summary = {
    "run_name": resolved_run_name,
    "model_name": CONFIG["model_name"],
    "num_examples": int(example_metrics["example_id"].nunique()),
    "samples_per_example": int(CONFIG["samples_per_example"]),
    "total_samples": int(len(sample_df)),
    "overall_parse_success_rate": float(sample_df["parse_success"].mean()),
    "overall_fraction_correct": float(sample_df["is_correct"].mean()),
    "mean_example_fraction_correct": float(example_metrics["fraction_correct"].mean()),
    "mean_example_entropy_bits": float(example_metrics["final_answer_entropy_bits"].mean()),
    "mean_example_reasoning_tokens": float(example_metrics["mean_reasoning_tokens"].mean()),
}

overall_summary_path = RUN_DIR / "overall_summary.json"
overall_summary_path.write_text(json.dumps(overall_summary, indent=2), encoding="utf-8")

print(f"Example metrics saved to: {example_metrics_path}")
print(f"Overall summary saved to: {overall_summary_path}")
display(example_metrics.head(3))
display(JSON(overall_summary))


## 7. Quick inspection tables

These views are handy for debugging high-entropy examples, low-accuracy examples, and individual sample outputs.


In [ ]:
print("Highest-entropy examples")
display(
    example_metrics.sort_values(
        ["final_answer_entropy_bits", "fraction_correct"],
        ascending=[False, True],
    ).head(10)
)

print("Lowest-accuracy examples")
display(
    example_metrics.sort_values(
        ["fraction_correct", "final_answer_entropy_bits"],
        ascending=[True, False],
    ).head(10)
)

print("Sample-level view for the highest-entropy example")
highest_entropy_example_id = int(
    example_metrics.sort_values("final_answer_entropy_bits", ascending=False).iloc[0]["example_id"]
)
display(
    sample_df.loc[sample_df["example_id"] == highest_entropy_example_id][
        [
            "example_id",
            "sample_id",
            "final_answer",
            "final_answer_normalized",
            "is_correct",
            "parse_success",
            "reasoning_tokens",
            "raw_completion",
        ]
    ].head(10)
)


## 8. Interactive viewer

Use the widgets below to choose a problem, filter to correct or incorrect generations, and inspect the exact prompt together with a selected generation.


In [ ]:
import html
import ipywidgets as widgets

if "sample_df" not in globals() or sample_df.empty:
    sample_df = load_existing_records(SAMPLE_JSONL_PATH)

if sample_df.empty:
    raise ValueError("No sample records found. Run the sampling cell first.")

if "selected_examples_path" not in globals():
    selected_examples_path = RUN_DIR / "selected_examples.csv"

if "example_metrics_path" not in globals():
    example_metrics_path = RUN_DIR / "example_metrics.csv"

if "selected_examples" not in globals() or selected_examples.empty:
    selected_examples = pd.read_csv(selected_examples_path)

if "example_metrics" not in globals() or example_metrics.empty:
    example_metrics = pd.read_csv(example_metrics_path)

if "tokenizer" not in globals():
    tokenizer = AutoTokenizer.from_pretrained(
        CONFIG["model_name"],
        trust_remote_code=CONFIG["trust_remote_code"],
    )

viewer_df = sample_df.copy()

for missing_col in ["parse_error", "matched_final_answer_block"]:
    if missing_col not in viewer_df.columns:
        viewer_df[missing_col] = None


def as_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes"}


def truncate_text(text: Any, max_chars: int = 100) -> str:
    text = str(text).strip()
    if len(text) <= max_chars:
        return text
    return text[: max_chars - 3] + "..."


def render_pre_block(title: str, body: Any) -> str:
    safe_title = html.escape(title)
    safe_body = html.escape("" if body is None else str(body))
    return (
        "<div style='border:1px solid #d0d7de; border-radius:8px; margin:8px 0; overflow:hidden;'>"
        f"<div style='background:#f6f8fa; padding:8px 12px; font-weight:600;'>{safe_title}</div>"
        f"<pre style='white-space:pre-wrap; word-wrap:break-word; margin:0; padding:12px; font-family:monospace; font-size:12px;'>{safe_body}</pre>"
        "</div>"
    )


viewer_df["example_id"] = viewer_df["example_id"].astype(int)
viewer_df["sample_id"] = viewer_df["sample_id"].astype(int)
viewer_df["is_correct"] = viewer_df["is_correct"].map(as_bool)

selected_examples = selected_examples.copy()
selected_examples["example_id"] = selected_examples["example_id"].astype(int)

example_lookup = selected_examples.set_index("example_id", drop=False).sort_index()
metrics_lookup = example_metrics.set_index("example_id", drop=False).sort_index()


def build_example_label(example_id: int) -> str:
    example_row = example_lookup.loc[example_id]
    metric_row = metrics_lookup.loc[example_id]
    return (
        f"example {example_id} | idx {int(example_row['dataset_index'])} | "
        f"acc={metric_row['fraction_correct']:.2f} | "
        f"H={metric_row['final_answer_entropy_bits']:.2f} | "
        f"{truncate_text(example_row['question'], max_chars=90)}"
    )


def filtered_rows(example_id: int, correctness_mode: str) -> pd.DataFrame:
    subset = viewer_df.loc[viewer_df["example_id"] == example_id].sort_values("sample_id")
    if correctness_mode == "correct":
        subset = subset.loc[subset["is_correct"]]
    elif correctness_mode == "incorrect":
        subset = subset.loc[~subset["is_correct"]]
    return subset


def build_generation_options(example_id: int, correctness_mode: str) -> list[tuple[str, int]]:
    subset = filtered_rows(example_id, correctness_mode)
    options: list[tuple[str, int]] = []
    for _, row in subset.iterrows():
        answer_text = truncate_text(row.get("final_answer", None), max_chars=30)
        parse_error = row.get("parse_error", None)
        parse_tag = "parse_error" if pd.notna(parse_error) and str(parse_error).strip() else "parsed"
        correctness_tag = "correct" if row["is_correct"] else "incorrect"
        label = (
            f"sample {int(row['sample_id'])} | {correctness_tag} | "
            f"answer={answer_text} | {parse_tag}"
        )
        options.append((label, int(row["sample_id"])))
    return options


example_selector = widgets.Dropdown(
    options=[(build_example_label(example_id), int(example_id)) for example_id in example_lookup.index],
    description="Example:",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "80px"},
)

correctness_selector = widgets.ToggleButtons(
    options=[
        ("All", "all"),
        ("Correct only", "correct"),
        ("Incorrect only", "incorrect"),
    ],
    description="Filter:",
    style={"description_width": "80px"},
)

generation_selector = widgets.Dropdown(
    options=[],
    description="Generation:",
    layout=widgets.Layout(width="100%"),
    style={"description_width": "80px"},
)

summary_html = widgets.HTML()
prompt_html = widgets.HTML()
generation_html = widgets.HTML()


def render_view(*_args: Any) -> None:
    example_id = int(example_selector.value)
    sample_id = generation_selector.value
    example_row = example_lookup.loc[example_id]
    metric_row = metrics_lookup.loc[example_id]
    prompt_text = render_prompt(
        question=example_row["question"],
        tokenizer=tokenizer,
        use_chat_template=CONFIG["use_chat_template"],
    )

    summary_html.value = (
        "<div style='padding:8px 0;'>"
        f"<b>Example {example_id}</b><br>"
        f"<b>Dataset index:</b> {int(example_row['dataset_index'])}<br>"
        f"<b>Gold answer:</b> {html.escape(str(example_row['gold_final_answer']))}<br>"
        f"<b>Fraction correct:</b> {metric_row['fraction_correct']:.3f}<br>"
        f"<b>Entropy:</b> {metric_row['final_answer_entropy_bits']:.3f} bits"
        "</div>"
    )
    prompt_html.value = render_pre_block("Prompt", prompt_text)

    if sample_id is None:
        generation_html.value = render_pre_block(
            "Generation",
            "No generations match the current filter for this example.",
        )
        return

    selected_row = viewer_df.loc[
        (viewer_df["example_id"] == example_id) & (viewer_df["sample_id"] == int(sample_id))
    ].iloc[0]

    generation_summary = (
        f"sample_id={int(selected_row['sample_id'])}\n"
        f"is_correct={bool(selected_row['is_correct'])}\n"
        f"final_answer={selected_row.get('final_answer', None)}\n"
        f"normalized_final_answer={selected_row.get('final_answer_normalized', None)}\n"
        f"parse_success={selected_row.get('parse_success', None)}\n"
        f"parse_error={selected_row.get('parse_error', None)}\n"
        f"reasoning_tokens={selected_row.get('reasoning_tokens', None)}\n"
        "\n"
        "Raw completion:\n"
        f"{selected_row.get('raw_completion', '')}"
    )
    generation_html.value = render_pre_block("Generation", generation_summary)


def refresh_generation_options(*_args: Any) -> None:
    options = build_generation_options(int(example_selector.value), str(correctness_selector.value))
    generation_selector.options = options
    generation_selector.disabled = len(options) == 0
    generation_selector.value = options[0][1] if options else None
    render_view()


example_selector.observe(refresh_generation_options, names="value")
correctness_selector.observe(refresh_generation_options, names="value")
generation_selector.observe(render_view, names="value")

refresh_generation_options()

display(
    widgets.VBox(
        [
            example_selector,
            correctness_selector,
            generation_selector,
            summary_html,
            prompt_html,
            generation_html,
        ]
    )
)
